# 2. First-Visit Monte Carlo
### Updates Using the First Visit to a State per Episode

In **First-Visit Monte Carlo**, the return $G_t$ following state $s$ is incorporated into $V(s)$'s sample average **only for the first time state $s$ is visited** within a given episode.

#### Mathematical Properties:
1. **Unbiased Estimator**:
   Each return $G_t$ is an independent and identically distributed (i.i.d.) estimate of $V^\pi(s)$:
   $$\mathbb{E}[G_t \mid S_t = s] = V^\pi(s)$$
2. **Incremental Update Equation**:
   $$V(s) \leftarrow V(s) + \frac{1}{N(s)} \left[ G_t - V(s) \right]$$
   where $[G_t - V(s)]$ is the estimation error.
3. **Variance**:
   Standard error of the estimate decreases with rate $1 / \sqrt{N(s)}$.


In [ ]:
# Step 1: Import Libraries
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict


In [ ]:
# Step 2: Define a Pure Python Blackjack Environment (Sutton & Barto Ch. 5)
# State: (player_sum: 12..21, dealer_showing_card: 1..10, usable_ace: bool)
# Actions: 0 = Stick, 1 = Hit

class BlackjackEnv:
    def __init__(self):
        self.actions = [0, 1]  # 0: Stick, 1: Hit
        
    def _draw_card(self):
        card = np.random.randint(1, 14)
        return min(card, 10)  # Face cards (J, Q, K) have value 10; Ace is 1
        
    def _draw_hand(self):
        return [self._draw_card(), self._draw_card()]
        
    def _get_hand_value(self, hand):
        val = sum(hand)
        has_ace = 1 in hand
        if has_ace and val + 10 <= 21:
            return val + 10, True  # Usable Ace
        return val, False
        
    def reset(self, start_state=None, start_action=None):
        if start_state is not None:
            player_sum, dealer_card, usable_ace = start_state
            self.player_hand = [1, player_sum - 11] if usable_ace else [player_sum // 2, player_sum - player_sum // 2]
            self.dealer_hand = [dealer_card, self._draw_card()]
        else:
            self.player_hand = self._draw_hand()
            while self._get_hand_value(self.player_hand)[0] < 12:
                self.player_hand.append(self._draw_card())
            self.dealer_hand = self._draw_hand()
            
        p_val, usable_ace = self._get_hand_value(self.player_hand)
        return (p_val, self.dealer_hand[0], usable_ace)
        
    def step(self, action):
        if action == 1:  # Hit
            self.player_hand.append(self._draw_card())
            p_val, usable_ace = self._get_hand_value(self.player_hand)
            if p_val > 21:
                return (p_val, self.dealer_hand[0], usable_ace), -1.0, True  # Bust
            return (p_val, self.dealer_hand[0], usable_ace), 0.0, False
        else:  # Stick: Dealer plays
            while True:
                d_val, _ = self._get_hand_value(self.dealer_hand)
                if d_val >= 17:
                    break
                self.dealer_hand.append(self._draw_card())
                
            p_val, usable_ace = self._get_hand_value(self.player_hand)
            d_val, _ = self._get_hand_value(self.dealer_hand)
            
            if d_val > 21 or p_val > d_val:
                reward = 1.0  # Player wins
            elif p_val == d_val:
                reward = 0.0  # Draw
            else:
                reward = -1.0 # Dealer wins
                
            return (p_val, self.dealer_hand[0], usable_ace), reward, True

env = BlackjackEnv()
print("Blackjack environment initialized.")


In [ ]:
# Step 3: Implement First-Visit MC with Incremental Updates
def first_visit_mc(env, policy, num_episodes=50000, gamma=1.0):
    V = defaultdict(float)
    N = defaultdict(int)
    
    for ep in range(num_episodes):
        # Generate an episode
        episode = []
        state = env.reset()
        while True:
            action = policy(state)
            next_state, reward, done = env.step(action)
            episode.append((state, action, reward))
            state = next_state
            if done:
                break
                
        visited_states = set()
        G = 0.0
        
        # Traverse episode in reverse order
        for t in reversed(range(len(episode))):
            s, a, r = episode[t]
            G = gamma * G + r
            
            # Check if this step is the first visit
            if s not in [step[0] for step in episode[:t]]:
                N[s] += 1
                # Incremental mean update: V(s) += (G - V(s)) / N(s)
                V[s] += (G - V[s]) / N[s]
                
    return V, N

# Policy: Stick on 19 or 20 or 21, hit otherwise
target_policy = lambda s: 0 if s[0] >= 19 else 1
V_first, N_first = first_visit_mc(env, target_policy, num_episodes=60000)

sample_state = (20, 10, False) # Player 20, Dealer 10, No Ace
print(f"State {sample_state}: Value = {V_first[sample_state]:.4f}, Visits = {N_first[sample_state]}")


In [ ]:
# Step 4: Sample Value Convergence vs Number of Episodes
episodes_checkpoints = [1000, 5000, 10000, 20000, 40000, 60000]
values_over_time = []

for ep in episodes_checkpoints:
    v_tmp, _ = first_visit_mc(env, target_policy, num_episodes=ep)
    values_over_time.append(v_tmp[sample_state])

plt.figure(figsize=(8, 4))
plt.plot(episodes_checkpoints, values_over_time, marker='o', color='teal', lw=2)
plt.xlabel("Number of Episodes")
plt.ylabel(f"Estimated Value of {sample_state}")
plt.title("First-Visit MC Value Estimate Convergence")
plt.grid(True)
plt.show()


### Summary
- Purely unbiased estimate of $V^\pi(s)$.
- Widely studied due to straightforward statistical properties.
